# Exploring the Dataset: Building the System CPU Metrics Table

**Goal:** Understand the structure of the `system.cpu` metricbeat logs, identify key fields, and map them to the `system_cpu_events` database table.

**Source file:** `gather/intranet_server/logs/2022-01-21-system_cpu.log`  
**What it contains:** CPU utilisation snapshots from the intranet server, collected every 45 seconds via metricbeat throughout 2022-01-21.

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below — everything else derives from it.

In [1]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path(r"C:\Users\ishaanshetty\DATA-201\russellmitchell")

cpu_log_path = DATASET_ROOT / "gather" / "intranet_server" / "logs" / "2022-01-21-system_cpu.log"

print(f"Dataset found at: {DATASET_ROOT}")


Dataset found at: C:\Users\ishaanshetty\DATA-201\russellmitchell


## 1. Load the Raw CPU Log File

The file is a JSON Lines document — each line is one metricbeat CPU snapshot with deeply nested fields. We flatten it with `json_normalize` and rename the key fields to clean column names.

In [2]:
import json
import pandas as pd

raw_records = []
with open(cpu_log_path) as f:
    for line in f:
        raw_records.append(json.loads(line.strip()))

df_raw = pd.json_normalize(raw_records)

# Extract and rename the columns we care about
df = df_raw[[
    '@timestamp', 'host.name',
    'system.cpu.total.pct', 'system.cpu.user.pct', 'system.cpu.system.pct',
    'system.cpu.idle.pct',  'system.cpu.iowait.pct', 'system.cpu.steal.pct',
    'system.cpu.softirq.pct', 'system.cpu.cores',
    'event.duration', 'metricset.period'
]].copy()

df.columns = [
    'timestamp', 'host',
    'cpu_total_pct', 'cpu_user_pct', 'cpu_system_pct',
    'cpu_idle_pct', 'cpu_iowait_pct', 'cpu_steal_pct',
    'cpu_softirq_pct', 'cores',
    'event_duration_ns', 'metricset_period_ms'
]

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Loaded {len(df)} records')
print(f'\nColumns: {list(df.columns)}')
print(f'\nHost: {df["host"].iloc[0]}')
print(f'Date: {df["timestamp"].dt.date.iloc[0]}')
print(f'Time range: {df["timestamp"].min().strftime("%H:%M:%S")} UTC to {df["timestamp"].max().strftime("%H:%M:%S")} UTC')
print(f'Sampling interval: {df["metricset_period_ms"].iloc[0]/1000:.0f}s  |  CPU cores: {int(df["cores"].iloc[0])}')


Loaded 1920 records

Columns: ['timestamp', 'host', 'cpu_total_pct', 'cpu_user_pct', 'cpu_system_pct', 'cpu_idle_pct', 'cpu_iowait_pct', 'cpu_steal_pct', 'cpu_softirq_pct', 'cores', 'event_duration_ns', 'metricset_period_ms']

Host: intranet-server
Date: 2022-01-21
Time range: 00:00:22 UTC to 23:59:37 UTC
Sampling interval: 45s  |  CPU cores: 1


## 2. Examine a Single Record

Before flattening, each record is a deeply nested JSON object from metricbeat. Here is what the raw structure looks like.

In [3]:
import json

print('Raw data for record 0:\n')
print(json.dumps(raw_records[0], indent=2))


Raw data for record 0:

{
  "agent": {
    "hostname": "intranet-server",
    "name": "intranet-server",
    "id": "d8e6f857-ec88-4cf5-bc5e-7b72a6fd1e33",
    "ephemeral_id": "c6b20ae2-38c0-4ada-8be2-2d607fddf7e9",
    "version": "7.13.2",
    "type": "metricbeat"
  },
  "service": {
    "type": "system"
  },
  "event": {
    "module": "system",
    "dataset": "system.cpu",
    "duration": 630326
  },
  "@version": "1",
  "metricset": {
    "period": 45000,
    "name": "cpu"
  },
  "host": {
    "cpu": {
      "pct": 0.0678
    },
    "name": "intranet-server"
  },
  "ecs": {
    "version": "1.9.0"
  },
  "@timestamp": "2022-01-21T00:00:22.284Z",
  "tags": [
    "beats_input_raw_event"
  ],
  "system": {
    "cpu": {
      "iowait": {
        "pct": 0.0002,
        "norm": {
          "pct": 0.0002
        }
      },
      "steal": {
        "pct": 0.0007,
        "norm": {
          "pct": 0.0007
        }
      },
      "irq": {
        "norm": {
          "pct": 0
        },
       

### What do these fields mean?

| Field | What It Is | Example |
|-------|------------|--------|
| `@timestamp` | When metricbeat collected this sample | `2022-01-21T00:00:22.284Z` |
| `system.cpu.total.pct` | Total CPU usage across all states | `0.0678` (6.8%) |
| `system.cpu.user.pct` | Time running user-space processes | `0.0283` |
| `system.cpu.system.pct` | Time in kernel / system calls | `0.0386` |
| `system.cpu.idle.pct` | Time the CPU was idle | `0.932` |
| `system.cpu.iowait.pct` | Time waiting for I/O to complete | `0.0002` |
| `system.cpu.steal.pct` | Time stolen by the hypervisor | `0.0007` |
| `event.duration` | How long metricbeat took to collect (ns) | `630326` |
| `metricset.period` | Configured collection interval (ms) | `45000` |

## 3. Field Schema

After flattening and renaming, the working DataFrame has these columns. All `pct` values are expressed as a fraction of 1.0 (e.g. 0.06 = 6%).

In [4]:
schema_rows = [
    ('timestamp',           'datetime64[ns, UTC]', 'Metricbeat collection timestamp'),
    ('host',                'str',                 'Hostname of the monitored server'),
    ('cpu_total_pct',       'float64',             'Total CPU usage (all states combined)'),
    ('cpu_user_pct',        'float64',             'CPU time in user-space processes'),
    ('cpu_system_pct',      'float64',             'CPU time in kernel/system calls'),
    ('cpu_idle_pct',        'float64',             'CPU idle time'),
    ('cpu_iowait_pct',      'float64',             'CPU waiting on I/O'),
    ('cpu_steal_pct',       'float64',             'CPU stolen by hypervisor'),
    ('cpu_softirq_pct',     'float64',             'CPU handling software interrupts'),
    ('cores',               'int64',               'Number of logical CPU cores'),
    ('event_duration_ns',   'int64',               'Time metricbeat took to collect (ns)'),
    ('metricset_period_ms', 'int64',               'Configured collection interval (ms)'),
]
pd.DataFrame(schema_rows, columns=['field','type','description'])


,field,type,description
0,timestamp,"datetime64[ns, UTC]",Metricbeat collection timestamp
1,host,str,Hostname of the monitored server
2,cpu_total_pct,float64,Total CPU usage (all states combined)
3,cpu_user_pct,float64,CPU time in user-space processes
4,cpu_system_pct,float64,CPU time in kernel/system calls
5,cpu_idle_pct,float64,CPU idle time
6,cpu_iowait_pct,float64,CPU waiting on I/O
7,cpu_steal_pct,float64,CPU stolen by hypervisor
8,cpu_softirq_pct,float64,CPU handling software interrupts
9,cores,int64,Number of logical CPU cores


## 4. Summary Statistics

Descriptive stats across the full day. Notice that the mean and median (50%) for `cpu_total_pct` are very close (~6–7%), indicating a stable baseline. The max of ~99% reveals the attack spike.

In [5]:
df[['cpu_total_pct','cpu_user_pct','cpu_system_pct',
    'cpu_idle_pct','cpu_iowait_pct']].describe().round(4)


,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_idle_pct,cpu_iowait_pct
stat,,,,,
count,1920.0000,1920.0000,1920.0000,1920.0000,1920.0000
mean,0.0690,0.0302,0.0359,0.9295,0.0014
std,0.0622,0.0318,0.0124,0.0674,0.0078
min,0.0330,0.0179,0.0119,0.0000,0.0000
25%,0.0622,0.0270,0.0335,0.9322,0.0002
50%,0.0644,0.0282,0.0351,0.9348,0.0005
75%,0.0667,0.0294,0.0367,0.9372,0.0009
max,0.9984,0.7430,0.3104,0.9668,0.1443


## 5. Percentile Distribution

The p99 is still only 8.7%, showing that 99% of the day was normal. The spike sits entirely above the p99 threshold — it is a clear outlier.

In [6]:
pcts = [50, 75, 90, 95, 99, 100]
cols  = ['cpu_total_pct','cpu_user_pct','cpu_system_pct','cpu_iowait_pct']
rows  = []
for p in pcts:
    row = {'percentile': f'p{p}'}
    for col in cols:
        row[col] = round(df[col].quantile(p/100), 4)
    rows.append(row)
pd.DataFrame(rows)


,percentile,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_iowait_pct
0,p50,0.0644,0.0282,0.0351,0.0005
1,p75,0.0667,0.0294,0.0367,0.0009
2,p90,0.0697,0.0308,0.0386,0.0024
3,p95,0.0728,0.0324,0.0406,0.0033
4,p99,0.0874,0.0366,0.0518,0.0099
5,p100,0.9984,0.7430,0.3104,0.1443


## 6. CPU Spike Records

Records where `cpu_total_pct > 0.5` (50%). All 10 fall within a 7-minute window starting at **06:26 UTC**. The high `cpu_user_pct` indicates the spike is driven by user-space processes — consistent with the webshell command execution seen in the access log at the same time.

In [7]:
df_spike = df[df['cpu_total_pct'] > 0.5][
    ['timestamp','cpu_total_pct','cpu_user_pct','cpu_system_pct','cpu_iowait_pct']
].reset_index(drop=True)
df_spike


,timestamp,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_iowait_pct
0,2022-01-21 06:26:37.284000+00:00,0.9938,0.0436,0.1090,0.0062
1,2022-01-21 06:27:22.284000+00:00,0.9984,0.0222,0.0940,0.0016
2,2022-01-21 06:28:07.284000+00:00,0.9976,0.1272,0.1097,0.0024
3,2022-01-21 06:28:52.284000+00:00,0.8904,0.7430,0.1217,0.1096
4,2022-01-21 06:29:37.284000+00:00,0.8490,0.4901,0.2233,0.1443
5,2022-01-21 06:30:22.284000+00:00,0.9513,0.5289,0.1616,0.0404
6,2022-01-21 06:31:07.284000+00:00,0.9559,0.5489,0.1783,0.0334
7,2022-01-21 06:31:52.284000+00:00,0.8916,0.5182,0.2683,0.0999
8,2022-01-21 06:32:37.284000+00:00,0.8492,0.5054,0.2370,0.1432
9,2022-01-21 06:33:22.284000+00:00,0.8502,0.4841,0.3104,0.1437


## 7. Spike in Context

The same window with 3 baseline samples before and after — shows the sharp rise from ~6% to ~99% and the clean return to normal once the attacker's commands finished.

In [8]:
spike_idx = df[df['cpu_total_pct'] > 0.5].index
ctx_start = max(0, spike_idx[0] - 3)
ctx_end   = min(len(df) - 1, spike_idx[-1] + 3)

df.iloc[ctx_start:ctx_end+1][
    ['timestamp','cpu_total_pct','cpu_user_pct','cpu_system_pct','cpu_iowait_pct']
].reset_index(drop=True)


,timestamp,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_iowait_pct
0,2022-01-21 06:24:22.284000+00:00,0.0624,0.0252,0.0362,0.0002
1,2022-01-21 06:25:07.284000+00:00,0.0749,0.0334,0.0407,0.1073
2,2022-01-21 06:25:52.284000+00:00,0.1162,0.0466,0.0679,0.1312
3,2022-01-21 06:26:37.284000+00:00,0.9938,0.0436,0.1090,0.0062
4,2022-01-21 06:27:22.284000+00:00,0.9984,0.0222,0.0940,0.0016
5,2022-01-21 06:28:07.284000+00:00,0.9976,0.1272,0.1097,0.0024
6,2022-01-21 06:28:52.284000+00:00,0.8904,0.7430,0.1217,0.1096
7,2022-01-21 06:29:37.284000+00:00,0.8490,0.4901,0.2233,0.1443
8,2022-01-21 06:30:22.284000+00:00,0.9513,0.5289,0.1616,0.0404
9,2022-01-21 06:31:07.284000+00:00,0.9559,0.5489,0.1783,0.0334


## 8. Mapping to the Database Schema

Here is how the flattened fields map to the planned `system_cpu_events` PostgreSQL table, followed by a preview of the first 10 rows as they will look once ingested.

| source_field | postgres_column | postgres_type | notes |
|---|---|---|---|
| timestamp | event_timestamp | TIMESTAMPTZ | Collection time from metricbeat |
| host | hostname | TEXT | Server identifier |
| cpu_total_pct | cpu_total_pct | NUMERIC(6,4) | 0.0–1.0 fraction |
| cpu_user_pct | cpu_user_pct | NUMERIC(6,4) | User-space fraction |
| cpu_system_pct | cpu_system_pct | NUMERIC(6,4) | Kernel fraction |
| cpu_idle_pct | cpu_idle_pct | NUMERIC(6,4) | Idle fraction |
| cpu_iowait_pct | cpu_iowait_pct | NUMERIC(6,4) | I/O wait fraction |
| cpu_steal_pct | cpu_steal_pct | NUMERIC(6,4) | Hypervisor steal fraction |
| cpu_softirq_pct | cpu_softirq_pct | NUMERIC(6,4) | Soft IRQ fraction |
| cores | cpu_cores | SMALLINT | Logical core count |

In [9]:
# Preview of the system_cpu_events table as it will look in PostgreSQL
db_preview = df[[
    'timestamp','cpu_total_pct','cpu_user_pct','cpu_system_pct',
    'cpu_idle_pct','cpu_iowait_pct'
]].head(10).reset_index(drop=True)
db_preview.insert(0, 'system_cpu_id', range(1, 11))
db_preview


,system_cpu_id,timestamp,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_idle_pct,cpu_iowait_pct
0,1,2022-01-21 00:00:22.284000+00:00,0.0678,0.0283,0.0386,0.9320,0.0002
1,2,2022-01-21 00:01:07.284000+00:00,0.0439,0.0211,0.0221,0.9559,0.0002
2,3,2022-01-21 00:01:52.284000+00:00,0.0500,0.0224,0.0267,0.9500,0.0000
3,4,2022-01-21 00:02:37.284000+00:00,0.0596,0.0262,0.0326,0.9404,0.0000
4,5,2022-01-21 00:03:22.284000+00:00,0.0610,0.0270,0.0331,0.9385,0.0005
5,6,2022-01-21 00:04:07.284000+00:00,0.0664,0.0279,0.0380,0.9331,0.0005
6,7,2022-01-21 00:04:52.284000+00:00,0.0626,0.0278,0.0339,0.9372,0.0002
7,8,2022-01-21 00:05:37.288000+00:00,0.0644,0.0277,0.0357,0.9354,0.0002
8,9,2022-01-21 00:06:22.284000+00:00,0.0640,0.0266,0.0355,0.9360,0.0000
9,10,2022-01-21 00:07:07.284000+00:00,0.0643,0.0273,0.0360,0.9348,0.0009
